In [1]:
import numpy as np
from scipy.signal import welch

# ======================
# Load data (from day 1)
# ======================
data = np.load("eeg_filtered.npy")
sfreq = 160  # Hz

# ======================
# 1) Segmentation
# ======================
window_size = 320   # 2 sec
step = 160          # 50% overlap

windows = []
for start in range(0, data.shape[1] - window_size, step):
    segment = data[:, start:start + window_size]
    windows.append(segment)

windows = np.array(windows)
print("Windows shape:", windows.shape)   # (n_windows, 64, 320)

# ======================
# 2) Feature Extraction
# ======================
bands = {
    "alpha": (8, 13),
    "beta": (13, 30)
}

def bandpower(signal, fs, band):
    f, Pxx = welch(signal, fs=fs, nperseg=256)
    idx = (f >= band[0]) & (f <= band[1])
    return np.mean(Pxx[idx])

features = []

for window in windows:
    feat = []
    for ch in window:
        # band powers
        feat.append(bandpower(ch, sfreq, bands["alpha"]))
        feat.append(bandpower(ch, sfreq, bands["beta"]))
        # simple stats
        feat.append(np.mean(ch))
        feat.append(np.std(ch))
    features.append(feat)

features = np.array(features)
print("Features shape:", features.shape)

# ======================
# 3) Save
# ======================
np.save("eeg_features.npy", features)
print("Feature extraction done ✅")

Windows shape: (59, 64, 320)
Features shape: (59, 256)
Feature extraction done ✅
